In [ ]:
# Habitat Extraction Script

# Import Required Libraries
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.features import rasterize
import geopandas as gpd
from pathlib import Path
import glob

In [ ]:
# Set directories (update these paths to your actual file locations)
# Within your input directory, you need:
# A GeoTIFF landcover raster with pixel values representing landcover types,
# A shapefile of protected lands (or whatever area you wish to extract),
# And a .csv file of landcover types which you wish to map.

Input = r"path\to\input" # Update this to your input directory 
Output = r"path\to\output" # Update this to your desired output directory

In [ ]:
# Find .csv file, geotiff raster and polygon shapefile in the input directory

csv_file = glob.glob(f"{Input}/*.csv")[0]
raster_file = glob.glob(f"{Input}/*.tif")[0]
shapefile = glob.glob(f"{Input}/*.shp")[0]

print(f"CSV file: {csv_file}")
print(f"Raster file: {raster_file}")
print(f"Shapefile: {shapefile}")

# Load the CSV file with landcover types
landcover_df = pd.read_csv(csv_file)
print(f"\nLandcover types:\n{landcover_df}")

In [ ]:
# Match the landcover types from the .csv file to the raster and extract
# This processes the raster in chunks to avoid memory issues
# This might take a while, so hang tight!

landcover_values = landcover_df['all'].values #Update column name if needed

# Read the raster in chunks for memory efficiency
from rasterio.windows import Window

with rasterio.open(raster_file) as src:
    raster_meta = src.meta.copy()
    raster_meta['driver'] = 'GTiff'

    # Update metadata for nodata value if not set
    if raster_meta.get('nodata') is None:
        raster_meta['nodata'] = -9999
    
    nodata = raster_meta['nodata']
    
    # Define chunk size (adjust based on available memory)
    chunk_size = 2048  
    
    # Calculate total number of chunks for progress tracking
    n_rows = (src.height + chunk_size - 1) // chunk_size
    n_cols = (src.width + chunk_size - 1) // chunk_size
    total_chunks = n_rows * n_cols
    
    print(f"Processing raster: {src.height} x {src.width} pixels in {total_chunks} chunks")
    
    # Prepare output file
    output_path_total = Path(Output) / "Top_Total.tif"
    
    with rasterio.open(output_path_total, 'w', **raster_meta) as dst:
        # Process raster in chunks
        chunk_num = 0
        for row in range(0, src.height, chunk_size):
            for col in range(0, src.width, chunk_size):
                chunk_num += 1
                
                # Define window
                window = Window(col, row, 
                              min(chunk_size, src.width - col),
                              min(chunk_size, src.height - row))
                
                # Read chunk
                raster_chunk = src.read(1, window=window)
                
                # Create mask for landcover types
                mask_array = np.isin(raster_chunk, landcover_values)
                
                # Extract matching landcover types
                extracted_chunk = np.where(mask_array, raster_chunk, nodata)
                
                # Write chunk to output
                dst.write(extracted_chunk, 1, window=window)
                
                # Print progress
                if chunk_num % 10 == 0 or chunk_num == total_chunks:
                    print(f"  Processed chunk {chunk_num}/{total_chunks} ({100*chunk_num/total_chunks:.1f}%)")
                
        print(f"Saved Top_Total raster to: {output_path_total}")

# Read unique values from output (sample a portion to avoid loading entire raster)
with rasterio.open(output_path_total) as src:
    sample_data = src.read(1, window=Window(0, 0, min(1024, src.width), min(1024, src.height)))
    print(f"Sample unique values in Top_Total: {np.unique(sample_data[sample_data != raster_meta['nodata']])}")

In [ ]:
# Overlay shapefile on raster and extract again
# This will also take forever, maybe do some chores

# Read the shapefile
gdf = gpd.read_file(shapefile)

# Set path to the new Top Total raster, should be saved in your output folder or can be copied from cell output above
output_path_total = Path(r"path\to\Top_Total.tif")  # Update this path if needed

# Process the Top_Total raster in chunks
with rasterio.open(output_path_total) as src:
    # Ensure the shapefile is in the same CRS as the raster
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)
    
    # Get shapefile geometries
    shapes = [geom for geom in gdf.geometry]
    
    nodata_value = src.nodata
    if nodata_value is None:
        nodata_value = -9999
        meta['nodata'] = nodata_value

    meta = src.meta.copy()
    meta['driver'] = 'GTiff'
    
    # Define chunk size
    chunk_size = 2048
    
    # Calculate total number of chunks for progress tracking
    n_rows = (src.height + chunk_size - 1) // chunk_size
    n_cols = (src.width + chunk_size - 1) // chunk_size
    total_chunks = n_rows * n_cols
    
    print(f"Processing protected areas: {src.height} x {src.width} pixels in {total_chunks} chunks")
    
    # Prepare output file
    output_path_protected = Path(Output) / "Top_Protected.tif"
    
    with rasterio.open(output_path_protected, 'w', **meta) as dst:
        # Process raster in chunks
        chunk_num = 0
        for row in range(0, src.height, chunk_size):
            for col in range(0, src.width, chunk_size):
                chunk_num += 1
                
                # Define window
                window = Window(col, row, 
                              min(chunk_size, src.width - col),
                              min(chunk_size, src.height - row))
                
                # Read chunk
                raster_chunk = src.read(1, window=window)
                
                # Get transform for this window
                window_transform = src.window_transform(window)
                
                # Create a binary mask for this chunk: 1 inside polygons, 0 outside
                polygon_mask = rasterize(
                    shapes,
                    out_shape=(window.height, window.width),
                    transform=window_transform,
                    fill=0,
                    default_value=1,
                    dtype=np.uint8
                )
                
                # Extract only values within polygons
                protected_chunk = np.where(polygon_mask == 1, raster_chunk, nodata_value)
                
                # Write chunk to output
                dst.write(protected_chunk, 1, window=window)
                
                # Print progress
                if chunk_num % 10 == 0 or chunk_num == total_chunks:
                    print(f"  Processed chunk {chunk_num}/{total_chunks} ({100*chunk_num/total_chunks:.1f}%)")
                
        print(f"Saved Top_Protected raster to: {output_path_protected}")

# Read unique values from output (sample a portion to avoid loading entire raster)
with rasterio.open(output_path_protected) as src:
    sample_data = src.read(1, window=Window(0, 0, min(1024, src.width), min(1024, src.height)))
    nodata_value = src.nodata
    print(f"Sample unique values in Top_Protected: {np.unique(sample_data[sample_data != nodata_value])}")

In [ ]:
# Calculate proportions of each landcover type that are protected
# Read from saved rasters in chunks to count pixels

# Set raster paths
output_path_protected = Path(r"path\to\Top_Protected.tif")
output_path_total = Path(r"path\to\Top_Total.tif")

# Initialize counters for each landcover type
pixel_counts = {}
for lc_value in landcover_values:
    pixel_counts[lc_value] = {'total': 0, 'protected': 0}

# Define chunk size
chunk_size = 2048

print("Counting pixels in rasters...")

# Open both rasters
with rasterio.open(output_path_total) as src_total, \
     rasterio.open(output_path_protected) as src_protected:
    
    # Calculate total chunks
    n_rows = (src_total.height + chunk_size - 1) // chunk_size
    n_cols = (src_total.width + chunk_size - 1) // chunk_size
    total_chunks = n_rows * n_cols
    
    # Process in chunks
    chunk_num = 0
    for row in range(0, src_total.height, chunk_size):
        for col in range(0, src_total.width, chunk_size):
            chunk_num += 1
            
            # Define window
            window = Window(col, row,
                          min(chunk_size, src_total.width - col),
                          min(chunk_size, src_total.height - row))
            
            # Read chunks from both rasters
            total_chunk = src_total.read(1, window=window)
            protected_chunk = src_protected.read(1, window=window)
            
            # Count pixels for each landcover type
            for lc_value in landcover_values:
                pixel_counts[lc_value]['total'] += np.sum(total_chunk == lc_value)
                pixel_counts[lc_value]['protected'] += np.sum(protected_chunk == lc_value)
            
            # Print progress
            if chunk_num % 10 == 0 or chunk_num == total_chunks:
                print(f"  Counted chunk {chunk_num}/{total_chunks} ({100*chunk_num/total_chunks:.1f}%)")

# Build results from pixel counts
results = []
for lc_value in landcover_values:
    total_pixels = pixel_counts[lc_value]['total']
    protected_pixels = pixel_counts[lc_value]['protected']
    proportion_protected = protected_pixels / total_pixels if total_pixels > 0 else 0
    
    results.append({
        'Landcover_Type': lc_value,
        'Pixels_Protected': int(protected_pixels),
        'Total_Pixels': int(total_pixels),
        'Proportion_Protected': proportion_protected
    })

# Create DataFrame and save to spreadsheet
results_df = pd.DataFrame(results)
output_csv = Path(Output) / "Top_Results.csv"
results_df.to_csv(output_csv, index=False)

print(f"\nSaved results to: {output_csv}")
print(f"\nResults:\n{results_df}")